# Quora Duplicate Detection – Model Training

Trains two models and saves them to `models/`.

**Simple model**: TF-IDF word cosine similarity → Logistic Regression  
**Improved model**: 18-feature vector (4 tokenisers + Levenshtein + WH-word flags) → Histogram-Based Gradient Boosting

> All function definitions live in `utils.py`. Do not define functions here.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

from utils import extract_simple_features, extract_improved_features

print('scikit-learn version:', sklearn.__version__)

## 1. Load Data

Primary path follows the deliverable spec (`$HOME/Datasets/QuoraQuestionPairs/`).  
Falls back to the local project path if running outside the teacher's environment.

In [ ]:
candidate_paths = [
    os.path.expanduser('~/Datasets/QuoraQuestionPairs/quora_data.csv'),
    os.path.join(os.path.dirname(os.path.abspath('__file__')),
                 '..', 'nlp_deliv1_materials', 'quora_data.csv'),
    '../nlp_deliv1_materials/quora_data.csv',
    './quora_data.csv',
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(
        'quora_data.csv not found. Place it at '
        '~/Datasets/QuoraQuestionPairs/quora_data.csv'
    )

print('Loading data from:', data_path)
quora_df = pd.read_csv(data_path)
print('Full dataset shape:', quora_df.shape)

## 2. Split Data

Fixed splits (random_state=123) as required by the deliverable guide.

In [ ]:
A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df, test_size=0.05, random_state=123)

print('train_df.shape =', train_df.shape)   # expected (291897, 6)
print('val_df.shape   =', val_df.shape)     # expected (15363, 6)
print('test_df.shape  =', test_df.shape)    # expected (16172, 6)
print('Duplicate rate (train):', round(train_df['is_duplicate'].mean(), 4))

In [ ]:
# Persist splits so reproduce_results.ipynb can load them without the raw CSV
train_df.to_csv('train_df.csv', index=False)
val_df.to_csv('val_df.csv',   index=False)
test_df.to_csv('test_df.csv',  index=False)
print('Data splits saved.')

## 3. Simple Model – TF-IDF Cosine Similarity + Logistic Regression

One feature per pair: cosine similarity between TF-IDF vectors.  
Baseline that captures lexical overlap without any hand-crafted engineering.

In [ ]:
# Fit TF-IDF on all training questions (q1 and q2 combined)
q1_train = train_df['question1'].fillna('').astype(str).values
q2_train = train_df['question2'].fillna('').astype(str).values
all_train_questions = np.concatenate([q1_train, q2_train])

tfidf_simple = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=50_000,
    sublinear_tf=True,
)
tfidf_simple.fit(all_train_questions)
print('TF-IDF vocabulary size:', len(tfidf_simple.vocabulary_))

In [ ]:
X_train_simple = extract_simple_features(train_df, tfidf_simple)
X_val_simple   = extract_simple_features(val_df,   tfidf_simple)

y_train = train_df['is_duplicate'].values
y_val   = val_df['is_duplicate'].values

print('Simple feature matrix shape:', X_train_simple.shape)

In [ ]:
simple_lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=123)
simple_lr.fit(X_train_simple, y_train)

y_prob_val_s = simple_lr.predict_proba(X_val_simple)[:, 1]
y_pred_val_s = simple_lr.predict(X_val_simple)
print('Simple model – validation metrics')
print(f'  ROC AUC:   {roc_auc_score(y_val, y_prob_val_s):.4f}')
print(f'  Precision: {precision_score(y_val, y_pred_val_s):.4f}')
print(f'  Recall:    {recall_score(y_val, y_pred_val_s):.4f}')

## 4. Improved Model – 18-Feature Vector + Histogram Gradient Boosting

Features are computed using four tokeniser strategies and several from-scratch
distance functions (Jaccard, overlap coefficient, BoW cosine, Levenshtein).

The classifier is `HistGradientBoostingClassifier`, which is much faster than
standard GBM while achieving comparable accuracy.

In [ ]:
# Word-level TF-IDF for cosine similarity feature
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=50_000,
    sublinear_tf=True,
)
tfidf_word.fit(all_train_questions)

# Character 2-gram TF-IDF for morphological similarity feature
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 3),
    min_df=5,
    max_features=50_000,
    sublinear_tf=True,
)
tfidf_char.fit(all_train_questions)

print('Word TF-IDF vocab size:', len(tfidf_word.vocabulary_))
print('Char TF-IDF vocab size:', len(tfidf_char.vocabulary_))

In [ ]:
print('Extracting training features (may take 3-5 minutes)...')
X_train_imp = extract_improved_features(train_df, tfidf_word, tfidf_char)
print('Training feature matrix shape:', X_train_imp.shape)

print('Extracting validation features...')
X_val_imp = extract_improved_features(val_df, tfidf_word, tfidf_char)
print('Done.')

In [ ]:
improved_gbm = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.1,
    max_leaf_nodes=63,
    random_state=123,
)
improved_gbm.fit(X_train_imp, y_train)

y_prob_val_i = improved_gbm.predict_proba(X_val_imp)[:, 1]
y_pred_val_i = improved_gbm.predict(X_val_imp)
print('Improved model – validation metrics')
print(f'  ROC AUC:   {roc_auc_score(y_val, y_prob_val_i):.4f}')
print(f'  Precision: {precision_score(y_val, y_pred_val_i):.4f}')
print(f'  Recall:    {recall_score(y_val, y_pred_val_i):.4f}')

## 5. Save Models

Skips saving if the `models/` directory already contains all artifacts  
(so running the notebook twice does not overwrite existing results).

In [ ]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

artifacts = {
    'tfidf_simple.joblib':   tfidf_simple,
    'simple_lr.joblib':      simple_lr,
    'tfidf_word.joblib':     tfidf_word,
    'tfidf_char.joblib':     tfidf_char,
    'improved_gbm.joblib':   improved_gbm,
}

all_exist = all(os.path.exists(os.path.join(MODELS_DIR, f)) for f in artifacts)

if all_exist:
    print('All model files already exist in models/ – skipping save.')
else:
    for fname, obj in artifacts.items():
        joblib.dump(obj, os.path.join(MODELS_DIR, fname))
        print(f'  Saved {fname}')
    print('All models saved.')